# 03 · A development plan and its financial limits

**What must this development earn, what cost can its income support, and can we pay for it on time?**

We continue with the same mixed-use property from [Notebook 02](design_to_value.ipynb).
First we set a construction plan. Then we declare a set of financial relationships
and solve it in two directions. Finally, we test whether the funding arrives before the bills.

This chapter adds `Formulation`, `Specification` and `Executor` to the Flow, Stream
and graph tools already introduced. Run it from a fresh kernel in the walkthrough
environment, with the `execution` extra installed. No earlier notebook state is needed.

## 1. The same property, now a development proposal

Building A contains hotel, residential, retail and parking space. Building B
contains residential, retail and parking space. Both depend on the shared basement
plinth and plant. The geometry shows **what** could be built; the development plan
will say **when** it is built and paid for.

In [1]:
from helpers.plot import show_geometry

from pathlib import Path
from datetime import date
from uuid import UUID, uuid4
import hashlib
import json
import math
import polars as pl
from IPython.display import IFrame, display

from rangekeeper.io import MemoryStore, json as codec
from rangekeeper.model import Model, Quantity
from rangekeeper.model.characteristics import value
from rangekeeper.model.content import decode
from rangekeeper.model.definitions import find_classifications
from rangekeeper.model.system import View, Hierarchy
from rangekeeper.model.duration import Span, Frequency, PeriodTiming
from rangekeeper.model.flux import Flow, Stream, MissingValueHandling
from rangekeeper.calculations import financial
from rangekeeper.calculations.series import ResamplingMethod
from helpers.plot import show_flows, show_plot, show_timeline, show_relations

pl.Config.set_thousands_separator(",")
pl.Config.set_fmt_str_lengths(80)
package = Path("data/mixed_use_property")
manifest = json.loads((package / "manifest.json").read_text())
assert (
    hashlib.sha256((package / "model.json").read_bytes()).hexdigest()
    == manifest["files"]["model.json"]["sha256"]
)
source_model = codec.read(package / "model.json", kind=Model)
source_before = codec.dumps(source_model)
legacy = json.loads((package / "assumptions.json").read_text())
plan = json.loads((package / "development.json").read_text())
output = Path("design-output/notebook-03")
output.mkdir(parents=True, exist_ok=True)
display(
    show_geometry("property", height=580)
)


Select the spatial relationships we used before. Each floor belongs to one package;
the plant and cores keep their original identities. The small lookup functions
below read recorded quantities; they do not assign financial rules.

In [2]:
whole_view = View(source_model)
spatial_class = find_classifications(
    source_model.definitions, code="spatiallyContains"
)[0].id
floor_class = find_classifications(source_model.definitions, code="floor")[0].id
hierarchy = Hierarchy.from_relationships(
    View(
        source_model,
        relationships=[
            r.id for r in whole_view.relationships if r.classification == spatial_class
        ],
    )
)
floors = [e for e in whole_view.entities if e.classification == floor_class]
phases = {p["name"]: p for p in plan["phases"]}
phase_ids = {UUID(p["object_id"]): name for name, p in phases.items()}


def phase_of(uid):
    while uid not in phase_ids:
        uid = hierarchy.parent(uid)
        if uid is None:
            raise ValueError("Object has no development package")
    return phase_ids[uid]


def quantity(entity, key, units):
    item = value(entity.characteristics, key)
    if item is None or item.quantity is None or item.quantity.units != units:
        raise ValueError(f"{entity.name}: missing or incompatible {key}")
    return item.quantity.magnitude


def use_of(entity):
    owner = (
        source_model.entity(hierarchy.parent(entity.id)) if entity in floors else entity
    )
    return decode(value(owner.characteristics, "use").content)


pl.DataFrame(
    [
        {
            "Package": name,
            "Object UUID": p["object_id"],
            "Floors": sum(phase_of(e.id) == name for e in floors),
            "GFA (m²)": sum(
                quantity(e, "gfa", "meter ** 2")
                for e in floors
                if phase_of(e.id) == name
            ),
        }
        for name, p in phases.items()
    ]
).with_columns(pl.col("GFA (m²)").round(2))


Package,Object UUID,Floors,GFA (m²)
str,str,i64,f64
"""Shared plinth""","""0a0a97ef-431d-471d-bc33-ace7e7df5fcb""",2,"12,773.74"
"""Building A""","""5ebb9736-5f18-4103-a5a4-040cd2e9b019""",21,"16,575.63"
"""Building B""","""a1e00481-e51e-47c5-b629-8b214c0900bd""",10,"13,436.88"


## 2. One base plan and one development budget

The **base plan** is the proposed sequence. The **base case** is its financial result
under specified assumptions, following the distinction in de Neufville and Geltner
(2018), {cite}`farevuu2018`.

Our teaching plan starts with an undeveloped site available for purchase in 2027.
It has no existing business or parking income. Shared works open in 2028, Building A
in 2030 and Building B in 2032. Opening assumes immediate stabilisation; lease-up is
deferred. The model does not choose these dates.

All new inputs are in [development.json](data/mixed_use_property/development.json).
They are illustrative, not market estimates. Notebook 02's rent and cost numbers
are rebased to 2027 **without** a market or inflation adjustment. Growth follows
one calendar from 2027; it does not restart when a building opens.

In [3]:
for p in phases.values():
    assert math.isclose(sum(p["payment_weights"]), 1.0)
    assert all(
        date.fromisoformat(phases[name]["open"]) <= date.fromisoformat(p["start"])
        for name in p["requires"]
    )
show_timeline(phases=list(phases.values()), path=output / "base-plan.html")


Construction is a new budget. Notebook 02's periodic CAPEX pays for asset renewal,
not initial development. Here, area rates include cores and plant, so we do not
also charge their volumes. Both soft costs and contingency are 10% of hard cost;
there is no cost escalation. Shared works enter once, under the plinth.

In [4]:
rates = plan["construction_AUD_per_m2"]
pl.DataFrame({"Use": list(rates), "Hard cost (AUD/m² GFA)": list(rates.values())})


Use,Hard cost (AUD/m² GFA)
str,i64
"""hotel""","3,000"
"""residential""","2,600"
"""retail""","2,400"
"""parking""","1,000"


In [5]:
worked_floor = source_model.entity(UUID(legacy["worked_floor_id"]))
worked_cost = quantity(worked_floor, "gfa", "meter ** 2") * rates[use_of(worked_floor)]
print(f"Worked hotel floor hard cost: AUD {worked_cost:,.2f}")
floor_costs = {
    e.id: quantity(e, "gfa", "meter ** 2") * rates[use_of(e)] for e in floors
}
hard_cost = {
    name: sum(cost for uid, cost in floor_costs.items() if phase_of(uid) == name)
    for name in phases
}
soft_fraction = plan["soft_cost_fraction_of_hard_cost"]
contingency_fraction = plan["contingency_fraction_of_hard_cost"]
package_cost = {
    name: cost * (1 + soft_fraction + contingency_fraction)
    for name, cost in hard_cost.items()
}
construction_cost = sum(package_cost.values())
land_cost = float(plan["land_cost_AUD"])
finance_allowance = float(plan["construction_finance_allowance_AUD"])
assert len(floor_costs) == 33
assert math.isclose(sum(hard_cost.values()), sum(floor_costs.values()))
pl.DataFrame(
    [
        {
            "Package": name,
            "Hard cost (AUD)": hard_cost[name],
            "Soft costs (AUD)": hard_cost[name] * soft_fraction,
            "Contingency (AUD)": hard_cost[name] * contingency_fraction,
            "Budget (AUD)": package_cost[name],
        }
        for name in phases
    ]
).with_columns(pl.col(pl.Float64).round(2))


Worked hotel floor hard cost: AUD 5,092,354.72


Package,Hard cost (AUD),Soft costs (AUD),Contingency (AUD),Budget (AUD)
str,f64,f64,f64,f64
"""Shared plinth""","12,773,744.21","1,277,374.42","1,277,374.42","15,328,493.05"
"""Building A""","45,818,888.97","4,581,888.9","4,581,888.9","54,982,666.77"
"""Building B""","32,887,640.26","3,288,764.03","3,288,764.03","39,465,168.32"


The land costs AUD 20 million at the start. A separate AUD 4 million allowance for
construction finance charges is paid at the end of 2031. This is a fixed allowance,
not a calculated interest schedule. We include it in the funding and SFFA tests;
we exclude it from **unlevered** project NPV to keep financing and property cash flows separate.

## 3. Turn the plan into dated cash flows

Use annual periods from 2027 through 2036, plus 2037 for the sale forecast.
Construction payments occur at year end, with equal payments over each building's
two construction years. Land is paid at the starting date.

In [6]:
base_date = date.fromisoformat(plan["base_date"])
holding = Span(
    name="Development and operations",
    start_inclusive=base_date,
    end_exclusive=date.fromisoformat(plan["horizon_end_exclusive"]),
)
forecast = Span(
    name="Including the sale forecast",
    start_inclusive=base_date,
    end_exclusive=date(2038, 1, 1),
)
periods = forecast.periods(frequency=Frequency.YEAR)
years = [p.start_inclusive.year for p in periods]

construction_by_package = {
    name: Flow.from_events(
        dates=[date(y, 12, 31) for y in p["payment_years"]],
        magnitudes=[-package_cost[name] * w for w in p["payment_weights"]],
        units="AUD",
    )
    for name, p in phases.items()
}
construction = (
    Stream(flows=construction_by_package)
    .resample(
        periods=periods,
        method=ResamplingMethod.SUM,
        missing=MissingValueHandling.ZERO,
    )
    .sum()
)
land = Flow.from_events(dates=[base_date], magnitudes=[-land_cost], units="AUD")


For each floor, keep the same area × efficiency × rent relationship and vacancy
deduction. Gate both income and OPEX by the package's opening date. Parking still
has zero rent and nonzero OPEX. As before, perimeter × floor height is a facade
proxy, not a measured external facade area.

In [7]:
use_inputs = legacy["use_assumptions"]


def floor_operations(entity):
    a = use_inputs[use_of(entity)]
    area = quantity(entity, "gfa", "meter ** 2")
    facade = quantity(entity, "perimeter", "meter") * quantity(entity, "ftf", "meter")
    first_egi = (
        area
        * a["efficiency"]
        * a["initial_rent_AUD_per_m2_per_year"]
        * (1 - a["vacancy"])
    )
    first_opex = (
        area * a["floor_opex_AUD_per_m2_per_year"]
        + facade * a["facade_opex_AUD_per_m2_per_year"]
    )
    opening = date.fromisoformat(phases[phase_of(entity.id)]["open"]).year
    egi = [
        (
            first_egi * (1 + a["annual_income_growth"]) ** (y - base_date.year)
            if y >= opening
            else 0
        )
        for y in years
    ]
    opex = [
        (
            first_opex * (1 + legacy["annual_opex_growth"]) ** (y - base_date.year)
            if y >= opening
            else 0
        )
        for y in years
    ]
    return (
        Flow.from_periods(periods=periods, magnitudes=egi, units="AUD"),
        Flow.from_periods(periods=periods, magnitudes=opex, units="AUD"),
    )


operations = {e.id: floor_operations(e) for e in floors}
egi = Stream(flows={str(uid): pair[0] for uid, pair in operations.items()}).sum()
opex = Stream(flows={str(uid): pair[1] for uid, pair in operations.items()}).sum()
noi = Stream(flows={"EGI": egi, "OPEX": opex}).sum()


Renewal CAPEX starts at the end of each package's fifth operating year and repeats
every five years. Retain the legacy 3.5% increase **per payment interval**. Include
the 33 floors and all three physical utilities, even when they produce no rent.

In [8]:
utilities = [
    source_model.entity(UUID(uid))
    for uid in (
        "7eec5208-fc0e-4dfd-bd0b-b1d226b2e97e",
        "cb5a02f7-b1b9-4a32-95b8-bbb70a946230",
        "938f4cf1-d307-4160-9538-d4bfa9583ad8",
    )
]
initial_renewal = {
    e.id: quantity(e, "gfa", "meter ** 2")
    * use_inputs[use_of(e)]["floor_capex_AUD_per_m2_per_payment"]
    for e in floors
}
initial_renewal.update(
    {
        e.id: quantity(e, "volume", "meter ** 3")
        * legacy["utility_capex_AUD_per_m3_per_payment"][use_of(e)]
        for e in utilities
    }
)


def renewal_flow(uid, initial):
    opening = date.fromisoformat(phases[phase_of(uid)]["open"]).year
    interval = plan["renewal_interval_years"]
    amounts = []
    for year in years:
        age = year - opening + 1
        amounts.append(
            initial
            * (1 + legacy["capex_growth_per_payment_interval"]) ** (age // interval - 1)
            if age > 0 and age % interval == 0
            else 0
        )
    return Flow.from_periods(periods=periods, magnitudes=amounts, units="AUD")


renewal = Stream(
    flows={
        str(uid): renewal_flow(uid, amount) for uid, amount in initial_renewal.items()
    }
).sum()
nacf = Stream(flows={"NOI": noi, "Renewal": renewal}).sum()
assert len(initial_renewal) == 36


In [9]:
valuation = plan["valuation"]
sale_amount = nacf.movements[-1].number / valuation["exit_capitalization_rate"]
sale = Flow.from_events(
    dates=[date.fromisoformat(valuation["sale_date"])],
    magnitudes=[sale_amount],
    units="AUD",
)
cashflows = (
    Stream(
        flows={
            "Land (start 2027)": land,
            "Construction": construction,
            "EGI": egi,
            "OPEX": opex,
            "Renewal": renewal,
            "Sale": sale,
        }
    )
    .trim(span=holding)
    .resample(
        periods=holding.periods(frequency=Frequency.YEAR),
        method=ResamplingMethod.SUM,
        missing=MissingValueHandling.ZERO,
    )
)
cashflows.display()


Period / date,Land (start 2027) [AUD],Construction [AUD],EGI [AUD],OPEX [AUD],Renewal [AUD],Sale [AUD]
2027,"-20,000,000.00","-15,328,493.05",0.00,0.00,0.00,0.00
2028,0.00,"-27,491,333.38",0.00,"-913,731.75",0.00,0.00
2029,0.00,"-27,491,333.38",0.00,"-945,712.36",0.00,0.00
2030,0.00,"-19,732,584.16","9,973,551.31","-4,631,385.40",0.00,0.00
2031,0.00,"-19,732,584.16","10,393,561.04","-4,793,483.89",0.00,0.00
2032,0.00,0.00,"18,193,426.66","-7,444,605.02","-1,281,793.00",0.00
2033,0.00,0.00,"19,072,982.53","-7,705,166.19",0.00,0.00
2034,0.00,0.00,"19,998,345.35","-7,974,847.01","-2,732,125.21",0.00
2035,0.00,0.00,"20,972,069.80","-8,253,966.66",0.00,0.00
2036,0.00,0.00,"21,996,861.68","-8,542,855.49","-1,905,174.39","258,141,522.61"


Zero-filled cells mean no scheduled payment, not an unknown amount. The table groups
land into 2027 for display; its discount exponent is still **zero**. All other
receipts and payments are at year end. Sale in 2036 capitalises 2037 **NACF**, including
that year's renewal costs, at 5%, retaining the convention from Notebook 02.

In [10]:
year_end_cash = cashflows.select(
    labels=["Construction", "EGI", "OPEX", "Renewal", "Sale"]
).sum()
discounted = financial.calculate_pv(
    flow=year_end_cash, rate=valuation["annual_discount_rate"], first_period=1
)
project_npv = discounted.total().magnitude - land_cost
pl.DataFrame(
    {
        "Measure": [
            "Land at start",
            "PV of year-end cash flows",
            "Unlevered project NPV",
        ],
        "AUD": [-land_cost, discounted.total().magnitude, project_npv],
    }
).with_columns(pl.col("AUD").round(2))


Measure,AUD
str,f64
"""Land at start""",-2e7
"""PV of year-end cash flows""","79,406,413.73"
"""Unlevered project NPV""","59,406,413.73"


In [11]:
show_flows(
    flows={
        "Construction": construction.trim(
            start=holding.start_inclusive, end=holding.end_exclusive
        ),
        "Operating NACF": nacf.trim(
            start=holding.start_inclusive, end=holding.end_exclusive
        ),
    },
    timing=PeriodTiming.LAST,
    title="Construction and operating cash flows · sale shown in the table",
    path=output / "development-cashflows.html",
)


This is a development NPV after land and construction costs. Notebook 02's value
was for an already completed asset with a different receipt calendar. They answer
different questions. The 7% discount rate is a teaching input; it is not derived
from the mortgage rate used next.

## 4. Declare the financial relationships once

Geltner et al.'s **simple financial feasibility analysis (SFFA)** asks what completed
income can support under permanent loan terms. The front-door procedure starts
with development costs and finds required rent. The back-door procedure starts
with rent and finds supportable cost (second edition, section 28.4.1, Exhibits
28-4 and 28-5, printed pages 771–773; {cite}`geltner2007`).

For a mixed-use project, one average rent would hide important differences. Use a
single **rent factor** across the existing hotel, residential and retail rents.
A factor of 1 is our reference basket; 1.1 increases each by 10%. Parking stays at
zero. Vacancies, areas and costs stay fixed. Use 2032, the first fully open year,
as the stabilised snapshot.

The SFFA cost includes the fixed finance allowance. At the threshold, we assume
completed value equals total cost and use 65% leverage and 1.25× debt coverage.
These are explicit assumptions, not proof of value or a lender's offer.

In [12]:
lending = plan["lending"]
payments_per_year = lending["payments_per_year"]
period_rate = lending["nominal_annual_mortgage_rate"] / payments_per_year
payment_count = lending["amortization_years"] * payments_per_year
mortgage_constant = (
    payments_per_year * period_rate / (1 - (1 + period_rate) ** -payment_count)
)
snapshot_index = years.index(plan["stabilised_year"])
reference_income = egi.movements[snapshot_index].number  # Amount for one complete year.
operating_cost = -opex.movements[snapshot_index].number  # Positive annual requirement.
pl.DataFrame(
    {
        "Input": [
            "Snapshot year",
            "Nominal interest rate",
            "Annual mortgage constant",
            "Leverage",
            "Debt coverage",
        ],
        "Value": [
            str(plan["stabilised_year"]),
            f"{lending['nominal_annual_mortgage_rate']:.0%}",
            f"{mortgage_constant:.4%} per year",
            f"{lending['leverage']:.0%}",
            f"{lending['coverage']:.2f}×",
        ],
    }
)


Input,Value
str,str
"""Snapshot year""","""2032"""
"""Nominal interest rate""","""6%"""
"""Annual mortgage constant""","""8.5972% per year"""
"""Leverage""","""65%"""
"""Debt coverage""","""1.25×"""


The annual mortgage constant includes principal repayment as well as interest;
it differs from 6%. Annual income and service rates have units **AUD/year**.
Cash-flow movements have units **AUD**, paid on a date or within a period.

Our six equations are:

1. Total cost = land cost + works cost (including the finance allowance).
2. Mortgage = leverage × total cost.
3. Annual debt service = mortgage constant × mortgage.
4. NOI = coverage × annual debt service.
5. Effective income = NOI + operating cost.
6. Effective income = rent factor × reference income.

An equation has two sides, but no permanent calculation direction. First give each
quantity a Value UUID and a Measure. Then express the relationships with references
to those Values. The unchanged geometry remains in the same Model revision lineage.

In [13]:
from rangekeeper.model import Metadata, Measure, Value, Update
from rangekeeper.schema.records import Reference, Assignment
from rangekeeper.schema.enums import ValueKind, SolutionStatus
from rangekeeper.model.expression.authoring import reference, equal, add, multiply
from rangekeeper.model.formulation.authoring import declare
from rangekeeper.specification import Specification, SpecificationRecord
from rangekeeper.run.execution import Executor

units = {
    "land_cost": "AUD",
    "works_cost": "AUD",
    "total_cost": "AUD",
    "mortgage": "AUD",
    "leverage": "dimensionless",
    "mortgage_constant": "1/year",
    "debt_service": "AUD/year",
    "coverage": "dimensionless",
    "noi": "AUD/year",
    "operating_cost": "AUD/year",
    "effective_income": "AUD/year",
    "rent_factor": "dimensionless",
    "reference_income": "AUD/year",
}
measures = {
    unit: Measure(id=uuid4(), code=f"sffa_{index}", name=unit, units=unit)
    for index, unit in enumerate(dict.fromkeys(units.values()))
}
values = {
    key: Value(
        id=uuid4(), key=key, kind=ValueKind.MEASUREMENT, measure=measures[unit].id
    )
    for key, unit in units.items()
}
value_ids = {key: item.id for key, item in values.items()}
symbols = {key: reference(Reference(target=uid)) for key, uid in value_ids.items()}


In [14]:
cost_equations = [
    (
        "cost",
        equal(
            left=symbols["total_cost"],
            right=add(left=symbols["land_cost"], right=symbols["works_cost"]),
        ),
    ),
    (
        "mortgage",
        equal(
            left=symbols["mortgage"],
            right=multiply(left=symbols["leverage"], right=symbols["total_cost"]),
        ),
    ),
    (
        "service",
        equal(
            left=symbols["debt_service"],
            right=multiply(
                left=symbols["mortgage_constant"], right=symbols["mortgage"]
            ),
        ),
    ),
]
income_equations = [
    (
        "coverage",
        equal(
            left=symbols["noi"],
            right=multiply(left=symbols["coverage"], right=symbols["debt_service"]),
        ),
    ),
    (
        "opex",
        equal(
            left=symbols["effective_income"],
            right=add(left=symbols["noi"], right=symbols["operating_cost"]),
        ),
    ),
    (
        "income",
        equal(
            left=symbols["effective_income"],
            right=multiply(
                left=symbols["rent_factor"], right=symbols["reference_income"]
            ),
        ),
    ),
]
relations = declare(
    id=uuid4(),
    name="Development feasibility",
    equations=cost_equations + income_equations,
    values=tuple(value_ids.values()),
)


In [15]:
property_object = source_model.entity(hierarchy.root)
financial_property = property_object.replace(
    characteristics=property_object.characteristics.replace(
        values=tuple(property_object.characteristics.values or ())
        + tuple(values.values())
    )
)
model = source_model.revise(
    Update(
        definitions=source_model.definitions.replace(
            measures=tuple(source_model.definitions.measures) + tuple(measures.values())
        ),
        system=source_model.system.replace(
            assemblies=tuple(
                financial_property if item.id == hierarchy.root else item
                for item in source_model.system.assemblies
            ),
            formulations=(relations,),
        ),
    )
)
assert codec.dumps(source_model) == source_before
assert model.metadata.previous == source_model.id


## 5. Change the question with a Specification

The Model states the relationships. The **Specification** declares which quantities
are fixed and which are unknown for this investigation. Values do not become fixed
merely because they have a recorded amount. With fixed coefficients, these equations
are linear in the unknowns and fit the current executor.

For the front door, fix land cost and solve for the rent factor and intermediate
amounts. `Executor` writes a new solved Model; it does not overwrite the source.

In [16]:
fixed = {
    "land_cost": land_cost,
    "works_cost": construction_cost + finance_allowance,
    "leverage": lending["leverage"],
    "mortgage_constant": mortgage_constant,
    "coverage": lending["coverage"],
    "operating_cost": operating_cost,
    "reference_income": reference_income,
}
front_spec = Specification(
    SpecificationRecord(
        metadata=Metadata(
            id=uuid4(), schema_version="0.7.0", name="Front door: required rent"
        ),
        model=model.id,
        assignments=tuple(
            Assignment(
                target=Reference(target=value_ids[key]),
                quantity=Quantity(magnitude=amount, units=units[key]),
            )
            for key, amount in fixed.items()
        ),
        unknowns=tuple(
            Reference(target=uid) for key, uid in value_ids.items() if key not in fixed
        ),
    )
)
store = MemoryStore()
store.put(model)
executor = Executor(store)
front_run = executor.execute(front_spec)
assert (
    front_run.report.status.solution is SolutionStatus.FEASIBLE
), front_run.report.to_data()
front_model = store.load_model(front_run.record.outputs[0])
required_factor = front_model.value(value_ids["rent_factor"]).quantity.magnitude


In [17]:
snapshot_year = plan["stabilised_year"]
pl.DataFrame(
    [
        {
            "Use": use,
            "Reference rent (AUD/m²/year)": a["initial_rent_AUD_per_m2_per_year"]
            * (1 + a["annual_income_growth"]) ** (snapshot_year - base_date.year),
            "Required rent (AUD/m²/year)": required_factor
            * a["initial_rent_AUD_per_m2_per_year"]
            * (1 + a["annual_income_growth"]) ** (snapshot_year - base_date.year),
        }
        for use, a in use_inputs.items()
    ]
).with_columns(pl.col(pl.Float64).round(2))


Use,Reference rent (AUD/m²/year),Required rent (AUD/m²/year)
str,f64,f64
"""hotel""",890.76,822.01
"""parking""",0.0,0.0
"""residential""",784.18,723.65
"""retail""","1,435.63","1,324.82"


Required rent is a threshold, not a forecast or a price the developer can command.
Reverse the question: **if the reference rents are achievable, what land price does
this threshold support?** Lock the factor at 1 and unlock land cost. The equations
and their Value UUIDs remain the same.

In [18]:
back_spec = front_spec.lock(
    target=value_ids["rent_factor"],
    quantity=Quantity(magnitude=1, units="dimensionless"),
    id=uuid4(),
    model=model,
).unlock(target=value_ids["land_cost"], id=uuid4(), model=model)
back_run = executor.execute(back_spec)
assert (
    back_run.report.status.solution is SolutionStatus.FEASIBLE
), back_run.report.to_data()
back_model = store.load_model(back_run.record.outputs[0])
supportable_land = back_model.value(value_ids["land_cost"]).quantity.magnitude
pl.DataFrame(
    {
        "Investigation": ["Front door", "Back door"],
        "Land cost (AUD)": [land_cost, supportable_land],
        "Rent factor": [required_factor, 1.0],
    }
).with_columns(pl.col(pl.Float64).round(4))


Investigation,Land cost (AUD),Rent factor
str,f64,f64
"""Front door""",2e7,0.9228
"""Back door""","40,103,550.834",1.0


In this diagram, edges mean **participates in**, not “calculates”. Equation numbers
match the list above. Switch investigations: fixed quantities are blue squares;
unknowns are rose circles. Only the two roles change.

In [19]:
names = {uid: key.replace("_", " ").capitalize() for key, uid in value_ids.items()}
names[value_ids["noi"]] = "NOI"
show_relations(
    formulation=relations,
    names=names,
    specifications={"Front door": front_spec, "Back door": back_spec},
    path=output / "relations.html",
)


A short check ties the inverse solve to its meaning: lock the front-door rent factor
in the back-door problem and recover the original land cost. Both Specifications
pin `model`, not either solved revision. To investigate a solved revision instead,
revise the Model pin explicitly first.

In [20]:
round_spec = back_spec.lock(
    target=value_ids["rent_factor"],
    quantity=Quantity(magnitude=required_factor, units="dimensionless"),
    id=uuid4(),
    model=model,
)
round_run = executor.execute(round_spec)
assert round_run.report.status.solution is SolutionStatus.FEASIBLE
round_model = store.load_model(round_run.record.outputs[0])
assert math.isclose(
    round_model.value(value_ids["land_cost"]).quantity.magnitude,
    land_cost,
    rel_tol=1e-9,
)
assert codec.dumps(source_model) == source_before
print(
    "The inverse solve recovers the original land cost; the source Model is unchanged."
)


The inverse solve recovers the original land cost; the source Model is unchanged.


## 6. What does each answer establish?

SFFA tests **loan support under stated assumptions**. Its cost/value proxy needs a
separate check. At the 2032 snapshot, compare cost with capitalised NOI; this is an
illustrative valuation cross-check, not the NACF terminal calculation in our DCF.
Debt coverage is not a developer's required return, and a feasible solve is not an
optimisation result.

In [21]:
snapshot_noi = noi.movements[snapshot_index].number
snapshot_value = snapshot_noi / valuation["exit_capitalization_rate"]
pl.DataFrame(
    {
        "Check": [
            "All-in SFFA cost",
            "2032 NOI / 5% (snapshot cross-check)",
            "Unlevered NPV at start 2027",
        ],
        "AUD": [
            land_cost + construction_cost + finance_allowance,
            snapshot_value,
            project_npv,
        ],
    }
).with_columns(pl.col("AUD").round(2))


Check,AUD
str,f64
"""All-in SFFA cost""","133,776,328.14"
"""2032 NOI / 5% (snapshot cross-check)""","214,976,432.91"
"""Unlevered NPV at start 2027""","59,406,413.73"


The snapshot and DCF differ in timing, renewal expenditure and finance treatment.
Do not require them to match. A negative residual land amount would mean that
reference income cannot cover the assumed non-land costs on these terms.

**Optional investigation: too many unknowns, incompatible requirements, or unsupported maths?**
Unlock both land and rent to allow many solutions. Lock an incompatible pair to
make the equations inconsistent. Unlock both leverage and total cost to introduce
an unknown product. These are three distinct outcomes; inspect the Run reports.

In [22]:
under_spec = front_spec.unlock(target=value_ids["land_cost"], id=uuid4(), model=model)
conflict_spec = front_spec.lock(
    target=value_ids["rent_factor"],
    quantity=Quantity(magnitude=required_factor + 0.1, units="dimensionless"),
    id=uuid4(),
    model=model,
)
nonlinear_spec = back_spec.unlock(target=value_ids["leverage"], id=uuid4(), model=model)
diagnostic_runs = {
    name: executor.execute(spec)
    for name, spec in {
        "Land and rent both unknown": under_spec,
        "Incompatible fixed pair": conflict_spec,
        "Unknown leverage × cost": nonlinear_spec,
    }.items()
}
pl.DataFrame(
    [
        {
            "Investigation": name,
            "Completion": run.report.status.completion.value,
            "Solution": run.report.status.solution.value,
            "Diagnostics": ", ".join(
                sorted(
                    {
                        d.code
                        for d in run.report.diagnostics
                        if d.severity.value != "info"
                    }
                )
            ),
        }
        for name, run in diagnostic_runs.items()
    ]
)


Investigation,Completion,Solution,Diagnostics
str,str,str,str
"""Land and rent both unknown""","""completed""","""feasible""","""underdetermined"""
"""Incompatible fixed pair""","""completed""","""infeasible""",""""""
"""Unknown leverage × cost""","""failed""","""not_assessed""","""settings_adjusted, unsupported_capability"""


The first case can return a feasible point without identifying a unique land value.
The second tests an inconsistent set of equations. The third exceeds the current
affine executor. An unsupported or interrupted calculation is not proof of
infeasibility. Choosing dates, uses or a preferred optimum requires a different
problem definition; it is not done by these inverse solves.

## 7. Can we pay the bills on time?

Now test a separate construction funding plan: AUD 50 million equity at purchase
and a committed AUD 65 million bridge facility. Draws arrive at the start of each
year; construction and operating payments occur at year end. No permanent loan
is available before 2032. The fixed AUD 4 million finance allowance falls due in 2031.

This test ends before permanent refinancing. It is a cash diagnostic, not an
equity valuation or an interest calculation. A negative balance is an **unfunded
amount**, not permission to overdraw the account. Annual timing also omits any
cash need inside a year.

In [23]:
funding = plan["funding"]
funding_years = list(range(base_date.year, funding["test_through_year"] + 1))
cash = funding["equity_AUD"] - land_cost
drawn = 0.0
funding_rows = []
for year in funding_years:
    index = years.index(year)
    opening_cash = cash
    draw = funding["draws_AUD_by_year"][str(year)]
    drawn += draw
    charge = (
        -finance_allowance
        if year == date.fromisoformat(plan["construction_finance_payment_date"]).year
        else 0
    )
    cash += (
        draw
        + construction.movements[index].number
        + nacf.movements[index].number
        + charge
    )
    funding_rows.append(
        {
            "Year": year,
            "Opening cash": opening_cash,
            "Draw": draw,
            "Construction": construction.movements[index].number,
            "Operating NACF": nacf.movements[index].number,
            "Finance charges": charge,
            "Closing cash": cash,
            "Undrawn facility": funding["bridge_limit_AUD"] - drawn,
        }
    )
funding_table = pl.DataFrame(funding_rows)
assert drawn <= funding["bridge_limit_AUD"]
funding_table.with_columns(pl.col(pl.Float64).round(2))


Year,Opening cash,Draw,Construction,Operating NACF,Finance charges,Closing cash,Undrawn facility
i64,f64,i64,f64,f64,f64,f64,f64
"2,027",3e7,"20,000,000","-15,328,493.05",0.0,0.0,"34,671,506.95",4.5e7
"2,028","34,671,506.95","15,000,000","-27,491,333.38","-913,731.75",0.0,"21,266,441.81",3e7
"2,029","21,266,441.81","15,000,000","-27,491,333.38","-945,712.36",0.0,"7,829,396.07",1.5e7
"2,030","7,829,396.07","15,000,000","-19,732,584.16","5,342,165.91",0.0,"8,438,977.82",0.0
"2,031","8,438,977.82",0,"-19,732,584.16","5,600,077.15",-4e6,"-9,693,529.19",0.0


In [24]:
funding_dates = [date(y, 12, 31) for y in funding_years]
show_flows(
    flows={
        "Cash before extra funding": Flow.from_events(
            dates=funding_dates,
            magnitudes=funding_table["Closing cash"].to_list(),
            units="AUD",
        ),
        "Undrawn committed facility": Flow.from_events(
            dates=funding_dates,
            magnitudes=funding_table["Undrawn facility"].to_list(),
            units="AUD",
        ),
    },
    title="Construction funding · year-end balances",
    path=output / "funding.html",
)


Cash on hand is not the only constraint. At the start of Building B, compare the
first payment with **all** costs already committed but unpaid. Include the
finance allowance. Exclude forecast operating receipts from this commitment test:
they are not secured funding. They remain visible in the cash forecast above.

In [25]:
commit_year = date.fromisoformat(phases["Building B"]["start"]).year
prior = next(row for row in funding_rows if row["Year"] == commit_year - 1)
committed_resources = prior["Closing cash"] + prior["Undrawn facility"]
first_payment = package_cost["Building B"] * phases["Building B"]["payment_weights"][0]
remaining_obligations = package_cost["Building B"] + finance_allowance
commitment_gap = remaining_obligations - committed_resources
peak_shortfall = max(0.0, -funding_table["Closing cash"].min())
pl.DataFrame(
    {
        "At start of Building B": [
            "Cash plus undrawn committed facility",
            "First construction payment",
            "All remaining committed costs",
            "Commitment funding gap",
        ],
        "AUD": [
            committed_resources,
            first_payment,
            remaining_obligations,
            commitment_gap,
        ],
    }
).with_columns(pl.col("AUD").round(2))


At start of Building B,AUD
str,f64
"""Cash plus undrawn committed facility""","22,829,396.07"
"""First construction payment""","19,732,584.16"
"""All remaining committed costs""","43,465,168.32"
"""Commitment funding gap""","20,635,772.25"


The first payment can fit while the complete commitment does not. Committed costs
are a forward check; we do not deduct them again when the cash schedule already
contains their payments. The commitment gap can exceed the forecast cash shortfall
because the latter includes future operating receipts.

Before proceeding, this plan needs more funding, lower costs or a changed sequence.
We have diagnosed the gap; the solver has not selected a remedy.

## 8. Keep the base case for the next question

These results describe one design, one fixed plan and one reference market basket.
The do-not-acquire alternative has zero cash flows and no retained property under
our stated starting condition. An option to control the site would need its own
price and obligations; free control is not assumed.

In [26]:
results = {
    "Required rent factor": required_factor,
    "Supportable land (AUD)": supportable_land,
    "Development NPV (AUD)": project_npv,
    "Peak cash shortfall (AUD)": peak_shortfall,
    "Commitment funding gap (AUD)": commitment_gap,
}
pl.DataFrame(
    {
        "Measure": [
            "Assumed land price",
            "SFFA supportable land",
            "Develop: unlevered NPV",
            "Do not acquire: NPV",
            "Peak construction cash shortfall",
            "Commitment funding gap",
        ],
        "AUD": [
            land_cost,
            supportable_land,
            project_npv,
            0.0,
            peak_shortfall,
            commitment_gap,
        ],
    }
).with_columns(pl.col("AUD").round(2))


Measure,AUD
str,f64
"""Assumed land price""",2e7
"""SFFA supportable land""","40,103,550.83"
"""Develop: unlevered NPV""","59,406,413.73"
"""Do not acquire: NPV""",0.0
"""Peak construction cash shortfall""","9,693,529.19"
"""Commitment funding gap""","20,635,772.25"


Three ideas carry forward:

- A financial relation stays the same when known and unknown roles change.
- A required income or supportable cost is conditional on its assumptions.
- A plan can have value and still lack funding at a commitment or payment date.

**Next:** keep the plan fixed and vary the markets. Notebook 04 will use the same
property, assumptions and common horizon to compare different market histories.
Later lessons will add timing choices and paid flexibility provisions.

The cell below saves this investigation's inputs and accepted outputs for inspection.
It does not make the next notebook depend on this kernel or choose a strategy.

In [27]:
records = output / "records" / str(model.id)
records.mkdir(parents=True, exist_ok=True)
for name, document in {
    "model": model,
    "front-specification": front_spec,
    "back-specification": back_spec,
    "front-run": front_run,
    "back-run": back_run,
    "front-model": front_model,
    "back-model": back_model,
}.items():
    codec.write(document, records / f"{name}.json")
(records / "base-plan.json").write_text(json.dumps(plan, indent=2) + "\n")
(records / "results.json").write_text(json.dumps(results, indent=2) + "\n")
funding_table.write_csv(records / "funding.csv")
assert codec.dumps(source_model) == source_before
print("Saved the base plan, Specifications, Runs and results under", records)


Saved the base plan, Specifications, Runs and results under design-output/notebook-03/records/5e5b4cd4-99f2-4c66-aca2-4c4cddab7e75


**Source and scope.** Geometry and identities come from the pinned legacy property
described in the [project brief](https://github.com/daniel-fink/rangekeeper/blob/7a0181774edcf29d363822b26cfb8739a467d11b/docs/guides/mixed-use-project.md). Operating
rules continue Notebook 02. Construction dates, prices and finance terms are
explicit teaching assumptions in `development.json`. SFFA follows the questions
in Geltner et al. (2007), section 28.4.1; the base-plan/base-case distinction follows
de Neufville and Geltner (2018). This example excludes tax, transaction costs,
pre-opening land holding costs, cost escalation and a lease-up period.